# 샘플링과학습_함수설명

각 설명 다음의 코드 셀을 Shift+Enter로 실행하세요. 이 노트북은 함수를 정의하고 내부 모듈을 준비합니다. 실제 전처리·학습 호출은 **00번 통합 노트북**에서 합니다. 내부 파일은 셀에서 자동 생성하므로 별도 Python 파일을 준비할 필요가 없습니다.

In [ ]:
from pathlib import Path
import sys, json
# 통합 노트북은 이 폴더를 작업 폴더로 지정합니다. 개별 실행도 같은 폴더에서 하세요.
WORKSPACE = Path.cwd().resolve()
RUNTIME = WORKSPACE / '_runtime'
(RUNTIME / 'gnn_pipeline').mkdir(parents=True, exist_ok=True)
(RUNTIME / 'gnn_pipeline' / '__init__.py').touch()
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))


## 실행 셀을 내부 모듈로 연결

아래 함수는 이 노트북의 코드 셀을 그대로 저장해 다음 단계에서 import할 수 있게 합니다.

In [ ]:
def export_cells(notebook_name, module_name):
    """이 노트북의 실제 코드 셀을 모아 내부 import용 모듈을 생성합니다."""
    notebook = json.loads((WORKSPACE / notebook_name).read_text(encoding='utf-8'))
    blocks = [cell['source'] for cell in notebook['cells']
              if cell['cell_type'] == 'code' and cell.get('metadata', {}).get('module') == module_name]
    source = '\n\n'.join(''.join(block) if isinstance(block, list) else block for block in blocks) + '\n'
    path = RUNTIME / 'gnn_pipeline' / (module_name + '.py')
    compile(source, str(path), 'exec')
    path.write_text(source, encoding='utf-8')
    print('내부 모듈 준비:', path.name)
    return path


## 라이브러리·상수 준비

다음 함수가 사용하는 라이브러리와 설정을 준비합니다.

In [ ]:
import numpy as np

## `allocate`

정상 표본 예산을 군집 크기 비례 75%와 균등 25%로 배분합니다. 존재하는 군집은 최소 한 건을 남깁니다.

**입력:** `counts`, `rate`, `equal_share`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `target`

In [ ]:
def allocate(counts, rate=0.01, equal_share=0.25):
    counts = np.asarray(counts, int)
    present = counts > 0
    budget = min(int(counts.sum()), max(int(present.sum()), int(np.ceil(counts.sum() * rate))))
    target = np.zeros(len(counts), int)
    target[present] = 1
    remaining = budget - target.sum()
    weights = (1 - equal_share) * counts / max(1, counts.sum()) + equal_share * present / max(1, present.sum())
    while remaining:
        capacity = counts - target
        active = capacity > 0
        w = weights * active
        w /= w.sum()
        extra = np.minimum(capacity, np.floor(remaining * w).astype(int))
        if not extra.sum():
            eligible = np.flatnonzero(active)
            ix = eligible[np.argsort(-w[eligible], kind='stable')[:remaining]]
            extra[ix] = 1
        target += extra
        remaining -= int(extra.sum())
    assert target.sum() == budget and (target <= counts).all() and (target[present] > 0).all()
    return target

## 다음 단계에 연결

실행한 코드 셀을 내부 모듈로 저장합니다. 내용을 수정했다면 이 셀을 다시 실행하고 통합 노트북의 커널을 재시작하세요.

In [ ]:
export_cells('40_샘플링과학습_함수설명.ipynb', 'allocation')

## 라이브러리·상수 준비

다음 함수가 사용하는 라이브러리와 설정을 준비합니다.

In [ ]:
"""Train six models sequentially; select checkpoints and thresholds on validation."""
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score
from gnn_pipeline import data
from gnn_pipeline.allocation import allocate
from gnn_pipeline.models import TransactionGNN
from gnn_pipeline.pipeline import read_graph, graph_dates, bind_run



## `prepare_samples`

학습 정상 거래에만 군집을 적합합니다. 이상 거래는 전부 보존하며, 선택한 정상 표본에 역확률 가중치를 부여합니다. 모든 모델이 같은 표본을 사용합니다.

**입력:** `settings`, `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `pd.DataFrame(rows)`

In [ ]:
def prepare_samples(settings, cfg):
    """Fit clusters on training negatives only; save common IDs/IPW for all models."""
    root = Path(cfg.output).parent / 'samples'
    root.mkdir(parents=True, exist_ok=True)
    dates = graph_dates(cfg, 'train')
    scaler = cluster = None
    if settings['sampling'] == 'cluster_under':
        reference = []
        for index, date in enumerate(dates):
            day = read_graph(cfg, date)
            positions = np.flatnonzero(day.labels == 0)
            rng = np.random.default_rng(cfg.seed + index)
            reference.append(day.target_x[rng.choice(positions, min(4096, len(positions)), replace=False)])
        reference = np.concatenate(reference)
        if len(reference) < settings['clusters']:
            raise ValueError('Fewer training negatives than clusters; lower clusters in config')
        scaler = StandardScaler().fit(reference)
        cluster = MiniBatchKMeans(n_clusters=settings['clusters'], random_state=cfg.seed,
                                  batch_size=8192, n_init=3).fit(scaler.transform(reference))
        np.savez(root / 'cluster_parameters.npz', mean=scaler.mean_, scale=scaler.scale_, centers=cluster.cluster_centers_)
    rows = []
    for index, date in enumerate(dates):
        day = read_graph(cfg, date)
        positive = np.flatnonzero(day.labels == 1)
        negative = np.flatnonzero(day.labels == 0)
        rng = np.random.default_rng(cfg.seed + index)
        selected, weights = list(positive), [1.] * len(positive)
        if settings['sampling'] == 'original':
            selected.extend(negative); weights.extend(np.ones(len(negative)))
        elif settings['sampling'] == 'random_under':
            size = min(len(negative), max(1, int(np.ceil(len(negative) * settings['negative_rate']))))
            if size:
                selected.extend(rng.choice(negative, size, replace=False))
                weights.extend([len(negative) / size] * size)
        elif len(negative):
            labels = cluster.predict(scaler.transform(day.target_x[negative]))
            counts = np.bincount(labels, minlength=settings['clusters'])
            budgets = allocate(counts, settings['negative_rate'])
            for group, size in enumerate(budgets):
                if size:
                    members = negative[labels == group]
                    selected.extend(rng.choice(members, int(size), replace=False))
                    weights.extend([len(members) / size] * int(size))
        order = np.argsort(selected)
        positions = np.asarray(selected, dtype=np.int64)[order]
        weights = np.asarray(weights, dtype=np.float64)[order]
        assert np.isclose(weights.sum(), len(day.labels))
        assert np.array_equal(np.sort(day.target_ids[positive]), np.sort(day.target_ids[positions][day.labels[positions] == 1]))
        table = pd.DataFrame({'position': positions, 'edge_id': day.target_ids[positions],
                              'y': day.labels[positions], 'sample_weight': weights})
        table.to_parquet(root / f'{date}.parquet', index=False)
        rows.append({'day': date, 'population': len(day.labels), 'sampled': len(table), 'positives': len(positive)})
    return pd.DataFrame(rows)

## `predict_validation`

전체 검증 거래의 예측값을 계산합니다. 가중치를 업데이트하지 않고 테스트 구간은 읽지 않습니다.

**입력:** `model`, `cfg`, `device`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `frame`

In [ ]:
def predict_validation(model, cfg, device):
    """Predict every validation transaction; never update model parameters."""
    model.eval()
    frames = []
    with torch.no_grad():
        for date in graph_dates(cfg, 'val'):
            day = read_graph(cfg, date)
            sampler = data.FixedNeighborSampler(day)
            scores = []
            for start in range(0, len(day.labels), cfg.target_batch_size):
                ix = np.arange(start, min(start + cfg.target_batch_size, len(day.labels)))
                batch = data.tensor_batch(sampler.batch(ix), device)
                scores.append(torch.sigmoid(model(*batch)).cpu().numpy())
            frames.append(pd.DataFrame({'edge_id': day.target_ids, 'day': date,
                                        'y': day.labels, 'score': np.concatenate(scores)}))
    frame = pd.concat(frames, ignore_index=True)
    if frame.y.nunique() != 2:
        raise ValueError('Validation requires both positive and negative transactions')
    data.finite('validation scores', frame.score.to_numpy())
    return frame

## `train_one`

모델 하나와 시드 하나를 학습합니다. 검증 AP가 가장 높은 가중치를 저장하고 검증 F1로 임계값을 선택합니다.

**입력:** `name`, `seed`, `cfg`, `device`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `json.loads((folder / 'complete.json').read_text())` / `result`

In [ ]:
def train_one(name, seed, cfg, device):
    """Train one model/seed; persist the best validation-AP checkpoint and F1 rule."""
    root = Path(cfg.output).parent
    folder = root / 'models' / f'{name}_seed{seed}'
    folder.mkdir(parents=True, exist_ok=True)
    if (folder / 'complete.json').exists():
        return json.loads((folder / 'complete.json').read_text())
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    model = TransactionGNN(name, hidden=cfg.hidden, heads=cfg.heads).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    best, stale, history = -np.inf, 0, []
    for epoch in range(cfg.epochs):
        model.train()
        for index, date in enumerate(graph_dates(cfg, 'train')):
            day = read_graph(cfg, date)
            table = pd.read_parquet(root / 'samples' / f'{date}.parquet')
            sampler = data.FixedNeighborSampler(day)
            order = np.random.default_rng(seed + epoch * 10000 + index).permutation(len(table))
            for start in range(0, len(order), cfg.target_batch_size):
                sample = table.iloc[order[start:start + cfg.target_batch_size]]
                ix = sample.position.to_numpy()
                optimizer.zero_grad(set_to_none=True)
                logits = model(*data.tensor_batch(sampler.batch(ix), device))
                labels = torch.as_tensor(day.labels[ix].copy(), dtype=torch.float32, device=device)
                weights = torch.as_tensor(sample.sample_weight.to_numpy(), dtype=torch.float32, device=device)
                losses = torch.nn.functional.binary_cross_entropy_with_logits(
                    logits, labels, pos_weight=torch.tensor(cfg.pos_weight, device=device), reduction='none')
                loss = (losses * weights).mean() * len(table) / len(day.labels)
                if not torch.isfinite(loss):
                    raise ValueError('Nonfinite training loss')
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5., error_if_nonfinite=True)
                optimizer.step()
        validation = predict_validation(model, cfg, device)
        ap = float(average_precision_score(validation.y, validation.score))
        history.append({'epoch': epoch + 1, 'validation_AP': ap})
        print(name, seed, history[-1], flush=True)
        if ap > best:
            best, stale = ap, 0
            torch.save({key: value.detach().cpu() for key, value in model.state_dict().items()}, folder / 'best.tmp')
            (folder / 'best.tmp').replace(folder / 'model.pt')
        else:
            stale += 1
            if stale >= cfg.patience:
                break
    model.load_state_dict(torch.load(folder / 'model.pt', map_location=device, weights_only=True))
    validation = predict_validation(model, cfg, device)
    threshold = data.select_threshold(validation)
    result = {'model': name, 'seed': seed, **data.evaluate_scores(validation, threshold), 'split': 'validation', 'test_used': False}
    validation.to_parquet(folder / 'validation_scores.parquet', index=False)
    pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
    data.json_write(folder / 'decision_rule.json', {'threshold': threshold, 'selected_on': 'validation F1'})
    data.json_write(folder / 'complete.json', result)
    return result

## `train_all`

공통 학습 표본을 만들고 선택한 모델·시드를 순서대로 학습한 뒤 비교 표를 저장합니다.

**입력:** `settings`, `cfg`.

**실행:** 이 셀에서는 함수를 정의합니다. 실제 작업은 인자를 넣어 호출할 때 시작됩니다.

**반환/결과:** `frame[['model', 'seed', 'AP', 'precision', 'recall', 'F1']]`

In [ ]:
def train_all(settings, cfg):
    """Prepare shared sampling and train the requested model/seed combinations."""
    root = bind_run(settings, cfg)
    torch.set_num_threads(cfg.threads)
    requested = settings.get('device', 'auto')
    device = ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested
    if device.startswith('cuda') and not torch.cuda.is_available():
        raise RuntimeError('CUDA unavailable; choose cpu or install a compatible PyTorch build')
    prepare_samples(settings, cfg).to_csv(root / 'sampling_summary.csv', index=False)
    results = [train_one(name, seed, cfg, device) for name in settings['models'] for seed in settings['seeds']]
    frame = pd.DataFrame(results)
    frame.drop(columns=['pattern_recall', 'confusion_matrix']).to_csv(root / 'comparison.csv', index=False)
    return frame[['model', 'seed', 'AP', 'precision', 'recall', 'F1']]

## 다음 단계에 연결

실행한 코드 셀을 내부 모듈로 저장합니다. 내용을 수정했다면 이 셀을 다시 실행하고 통합 노트북의 커널을 재시작하세요.

In [ ]:
export_cells('40_샘플링과학습_함수설명.ipynb', 'training')